In [1]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import joblib

# load and read the csv file(dataset)

In [2]:
data = pd.read_csv('spam.csv', encoding='latin-1')

## view your first five row data

In [3]:
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# change the data featurs from v1, v2 to 'label' & 'message'

In [4]:
new_data = data[['v1', 'v2']]
new_data.columns = ['label', 'message']

# view the new data to confirm the name we changed

In [5]:
new_data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
new_data['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

# convertin 'ham' to 0, 'spam' to 1

In [7]:
new_data['label'] = new_data['label'].map({'ham': 0, 'spam': 1})

In [8]:
new_data.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


# convert all the capital letter in the message all to smaller letter

In [9]:
def procees_text(text):
    new_text = text.lower()
    return  new_text

new_data['processed_message'] = new_data['message'].apply(procees_text)

# view the new message to confirm the converted capital letters

In [10]:
new_data.head()

,label,message,processed_message
0,0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,0,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro..."


# how to convert our sentence in message to numbers

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(new_data['processed_message'])

# Select the first 10 rows of X and convert from sparse matrix to dense array

In [12]:
X[0:10].toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(10, 8672))

# Extract the 'label' column from the DataFrame and store it as target variable y & Display y

In [13]:
y = new_data['label']
y

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: label, Length: 5572, dtype: int64

# Spliting your dataset into training and testing sets

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

 # Create SMOTE object to handle class imbalance, it balanced the feature and label data

In [15]:
smote = SMOTE(random_state=42)
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

In [16]:
log_model = LogisticRegression()
log_model.fit(x_train_smote, y_train_smote)
log_pred = log_model.predict(x_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))

Logistic Regression Accuracy: 0.9766816143497757


In [17]:
grad_model = GradientBoostingClassifier()
grad_model.fit(x_train_smote, y_train_smote)
grad_pred = grad_model.predict(x_test)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, grad_pred))

Gradient Boosting Accuracy: 0.9730941704035875


In [18]:
forest_model = RandomForestClassifier()
forest_model.fit(x_train_smote, y_train_smote) 
forest_pred = forest_model.predict(x_test)
print("Random Forest Accuracy:", accuracy_score(y_test, forest_pred))

Random Forest Accuracy: 0.979372197309417


In [19]:
ensemble_model = VotingClassifier(
    estimators=[
        ('log', log_model), 
        ('grad', grad_model), 
        ('forest', forest_model)],
    voting='soft')

ensemble_model.fit(x_train_smote, y_train_smote)
ensemble_pred = ensemble_model.predict(x_test)
print("Ensemble Model Accuracy:", accuracy_score(y_test, ensemble_pred))

Ensemble Model Accuracy: 0.9811659192825112


In [20]:
joblib.dump(vectorizer, 'sms_spam_vectorizer.pkl') # Save text vectorizer for future use
joblib.dump(log_model, 'sms_spam_logistic_model.pkl') # Save the logistic regression model
joblib.dump(grad_model, 'sms_spam_gradient_model.pkl') # Save the gradient boosting model  
joblib.dump(forest_model, 'sms_spam_random_forest_model.pkl') # Save the random forest model
joblib.dump(ensemble_model, 'sms_spam_model.pkl') # Save the trained model for future use


['sms_spam_model.pkl']

In [ ]:
gr